<a href="https://colab.research.google.com/github/ppavon/teaching-materials/blob/main/notebooks/cao-latency/cao-latency-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cao Formula (Best-Effort): Queue Delay Percentiles

**Course:** Planificación y gestión de redes (3rd year, Grado en Ingeniería Telemática)  
**University:** Universidad Politécnica de Cartagena (UPCT, Spain)

This notebook is an **educational resource** for the course *Network Planning and Management*.  
It allows students to interactively experiment with the **Cao formula** to estimate **high queueing delay percentiles** (P99 and P99.9) in **best-effort IP links**.

---

## 1) Problem under study

We aim to estimate the **extreme queueing delay** on a **single best-effort IP link** as a function of:

- Link utilization $u \in (0,1)$  
- Link capacity $\beta$ (bps)  
- Mean connection size $\gamma_b$ (bps per connection)  
- Mean number of active connections $c$

Key ideas:

- Higher **statistical multiplexing** $\Rightarrow$ lower delay  
- Higher **capacity** $\Rightarrow$ lower delay for the same $u$  
- As $u \rightarrow 1$, delays grow very rapidly  

---

## 2) Logit function

The **logit function** is a mathematical transformation that maps a probability  
$u \in (0,1)$ into an unbounded real value.

In this model, a base-2 logit is used:

$$
\logit_2(u) = \log_2\left(\frac{u}{1-u}\right)
$$

Important properties:

- Defined only for $0 < u < 1$  
- It is a **strictly increasing** function  
- As $u \rightarrow 0$, $\logit_2(u) \rightarrow -\infty$  
- As $u \rightarrow 1$, $\logit_2(u) \rightarrow +\infty$

This transformation is particularly suitable to model phenomena that  
**blow up close to saturation**, such as queueing delay in networks.

---

## 3) Cao model (2004)

Cao’s formula relates link utilization to a high delay percentile:

$$
\logit_2(u)
=
o
+
(o_c + o_{\tau\delta}) \log_2(c)
+
o_{\tau\delta} \log_2(\gamma_b \, \delta)
+
o_w \left(-\log_2\left(-\log_2(w)\right)\right)
$$

where:

- $u$: link utilization  
- $\delta$: delay (in seconds) associated with the selected percentile  
- $w = \Pr\{\text{delay} \ge \delta\}$  
  - P99 $\Rightarrow w = 0.01$  
  - P99.9 $\Rightarrow w = 0.001$  
- $c$: mean number of active connections  
- $\gamma_b$: mean connection size (bps per connection)  

---

## 4) Model parameters

Values used in this notebook (empirical fit by Cao):

- $o = -8.933$  
- $o_c = 0.420$  
- $o_{\tau\delta} = 0.444$  
- $o_w = 0.893$  

---

## 5) Application in this notebook

For each link capacity $\beta$ and each utilization value $u$:

1. Mean link traffic:

$$
\tau = u \, \beta
$$

2. Mean number of active connections:

$$
c = \frac{u \, \beta}{\gamma_b}
$$

3. The delay $\delta$ is obtained by solving Cao’s model.

4. $\delta$ (in milliseconds) is plotted versus link utilization $u$.

---

## 6) Educational objective

To understand the relationship between:

$$
\text{utilization}
\;\; \leftrightarrow \;\;
\text{multiplexing}
\;\; \leftrightarrow \;\;
\text{extreme delay}
$$

and to analyze its impact on best-effort IP network planning.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definition of the base-2 logit function
def logit2(u):
    return np.log2(u / (1 - u))

# Utilization range (avoiding 0 and 1)
u = np.linspace(0.001, 0.999, 500)

plt.figure(figsize=(6, 4))
plt.plot(u, logit2(u))

plt.xlabel("Link utilization $u$")
plt.ylabel(r"$\mathrm{logit}_2(u)$")
plt.title(r"Function $\mathrm{logit}_2(u) = \log_2\!\left(\frac{u}{1-u}\right)$")

plt.grid(True)
plt.show()



In [ ]:
# ===== Cell 1: imports =====
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
# ===== Cell 2: model parameters (from lecture notes) =====
# See slides "Network simulation & overprovisioning requirements estimation"
# (Cao 2004): fitted parameters
O_HAT   = -8.933
OC_HAT  =  0.420
OTD_HAT =  0.444
OW_HAT  =  0.893

def logit2(u: np.ndarray) -> np.ndarray:
    """log2(u/(1-u)) for u in (0,1)."""
    return np.log2(u / (1.0 - u))


def cao_delay_seconds(u: np.ndarray, beta_bps: float, gamma_b_bps: float, w: float) -> np.ndarray:
    """
    Computes delta (seconds) for each u by solving Cao's model.

    - u: array of utilizations (0..0.99)
    - beta_bps: link capacity (bps)
    - gamma_b_bps: mean connection size (bps per connection)
    - w: P(delay >= delta). For P99: 0.01. For P99.9: 0.001
    """
    u = np.asarray(u, dtype=float)

    # Avoid numerical issues at u = 0 or u = 1:
    # for u = 0 we will explicitly plot delta = 0.
    eps = 1e-12
    u_safe = np.clip(u, eps, 1.0 - eps)

    # c = (u * beta) / gamma_b
    c = (u_safe * beta_bps) / gamma_b_bps

    # Avoid log2(c) with c < 1 (limited physical interpretation, but numerically stable):
    # apply a soft minimum.
    c_safe = np.clip(c, 1.0, None)

    # w term:  -log2(-log2(w))
    w_term = (-np.log2(-np.log2(w)))

    rhs = (
        logit2(u_safe)
        - O_HAT
        - (OC_HAT + OTD_HAT) * np.log2(c_safe)
        - OTD_HAT * np.log2(gamma_b_bps)
        - OW_HAT * w_term
    )

    # o_{τδ} * log2(delta) = rhs  =>  log2(delta) = rhs / o_{τδ}
    log2_delta = rhs / OTD_HAT
    delta = 2.0 ** log2_delta  # seconds

    # Force delta = 0 when u = 0 exactly, as requested for the plot
    delta = np.where(u == 0.0, 0.0, delta)
    return delta


In [ ]:
# ===== Cell 3: widgets (controls) clear and vertically arranged =====

import ipywidgets as widgets
from IPython.display import display

title = widgets.HTML(
    "<h2>Cao Formula (Best-Effort): delay percentiles vs. utilization</h2>"
)

percentile = widgets.ToggleButtons(
    options=[("P99", "P99"), ("P99.9", "P99.9")],
    value="P99.9",
    description="Percentile:",
    style={"description_width": "140px"},
)

capacities_text = widgets.Text(
    value="1 10 40 100 400",
    description="Capacities (Gbps):",
    placeholder="e.g.: 1 10 40 100 400",
    layout=widgets.Layout(width="650px"),
    style={"description_width": "140px"},
)

# --- Mean connection size: slider + synchronized manual input ---
gamma_slider = widgets.FloatSlider(
    value=1.0,
    min=0.05,
    max=100.0,
    step=0.05,
    description="Mean connection size (Mbps):",
    readout=True,
    readout_format=".2f",
    style={"description_width": "240px"},
    layout=widgets.Layout(width="650px"),
)

gamma_text = widgets.BoundedFloatText(
    value=1.0,
    min=0.05,
    max=100.0,
    step=0.05,
    description="(exact value):",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="300px"),
)

# Synchronize both (frontend, instantaneous)
widgets.jslink((gamma_slider, "value"), (gamma_text, "value"))

gamma_box = widgets.HBox([gamma_slider, gamma_text])

# --- Y-axis limit: visualization only (slider + manual input) ---
ymax_slider = widgets.FloatSlider(
    value=500.0,
    min=10.0,
    max=500.0,
    step=10.0,
    description="Y-axis limit (ms) [visualization only]:",
    readout=True,
    readout_format=".0f",
    style={"description_width": "330px"},
    layout=widgets.Layout(width="650px"),
)

ymax_text = widgets.BoundedFloatText(
    value=500.0,
    min=10.0,
    max=500.0,
    step=10.0,
    description="(exact value):",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="300px"),
)

widgets.jslink((ymax_slider, "value"), (ymax_text, "value"))

ymax_box = widgets.HBox([ymax_slider, ymax_text])

# --- Option: clip values to the visualization limit (to avoid “overflow”) ---
clip_checkbox = widgets.Checkbox(
    value=True,
    description="Clip delays to the Y-axis limit (visualization only)",
)

btn = widgets.Button(
    description="Generate plot",
    button_style="primary",
    icon="line-chart",
    layout=widgets.Layout(width="220px")
)

out = widgets.Output()

sep = widgets.HTML("<hr style='margin:10px 0;'>")

ui = widgets.VBox([
    title,
    sep,
    percentile,
    sep,
    capacities_text,
    sep,
    gamma_box,
    sep,
    ymax_box,
    clip_checkbox,
    sep,
    btn,
    out
])

display(ui)


In [ ]:
# ===== Cell 4: callback to draw the plot =====

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

def parse_capacities_gbps(text: str):
    tokens = text.replace(",", " ").split()
    caps = []
    for t in tokens:
        try:
            caps.append(float(t))
        except ValueError:
            pass
    # remove duplicates while preserving order
    seen = set()
    caps2 = []
    for c in caps:
        if c > 0 and c not in seen:
            seen.add(c)
            caps2.append(c)
    return caps2

def on_click(_):
    with out:
        clear_output(wait=True)

        caps_gbps = parse_capacities_gbps(capacities_text.value)
        if not caps_gbps:
            print("⚠️ Unable to parse capacities. Valid example: 1 10 40 100 400")
            return

        # u: 0%..99% in steps of 1%
        u = np.arange(0.0, 1.0, 0.01)
        u = u[u <= 0.99]

        # percentile => w
        w = 0.01 if percentile.value == "P99" else 0.001

        # gamma_b in bps (synchronized slider/text value)
        gamma_b_bps = gamma_slider.value * 1e6

        # ymax (ms) for visualization only
        y_max_ms = ymax_slider.value

        plt.figure(figsize=(10, 5))

        for cap in caps_gbps:
            beta_bps = cap * 1e9
            delta_s = cao_delay_seconds(u, beta_bps, gamma_b_bps, w)
            delta_ms = 1e3 * delta_s

            # Optional: clip values to the visualization limit
            if clip_checkbox.value:
                delta_ms = np.clip(delta_ms, 0.0, y_max_ms)

            plt.plot(100*u, delta_ms, label=f"{cap:g} Gbps")

        plt.xlabel("Link utilization $u$ (%)")
        plt.ylabel(f"{percentile.value} delay (ms)")
        plt.title(rf"Cao best-effort — $\gamma_b$={gamma_slider.value:.2f} Mbps/conn")
        plt.grid(True)
        plt.ylim(0, y_max_ms)  # dynamic Y-axis limit (visualization only)
        plt.legend()
        plt.show()

btn.on_click(on_click)

# Auto-run once
on_click(None)
